# 🏟️ Análisis de Eventos Deportivos
**Herramientas:** Python · Pandas  
**Objetivo:** Procesar y analizar datos de eventos, aficionados y promociones para extraer insights clave que ayuden a mejorar la planificación de futuros eventos.


## 1. Importar librerías

In [1]:
import pandas as pd
import os

## 2. Cargar y limpiar datos

Cargamos los tres CSV en DataFrames independientes y eliminamos filas con valores nulos con `dropna()`.

In [2]:
try:
    df_eventos     = pd.read_csv("workspace/eventos.csv")
    df_aficionados = pd.read_csv("workspace/aficionados.csv")
    df_promociones = pd.read_csv("workspace/promociones.csv")
except FileNotFoundError as e:
    raise SystemExit(f"Archivo no encontrado: {e}")
except Exception as e:
    raise SystemExit(f"Error al cargar los datos: {e}")

df_eventos     = df_eventos.dropna()
df_aficionados = df_aficionados.dropna()
df_promociones = df_promociones.dropna()

# Verificamos que no quedan nulos
print("Nulos en eventos:     ", df_eventos.isnull().sum().sum())
print("Nulos en aficionados: ", df_aficionados.isnull().sum().sum())
print("Nulos en promociones: ", df_promociones.isnull().sum().sum())

Nulos en eventos:      0
Nulos en aficionados:  0
Nulos en promociones:  0


## 3. Combinación de DataFrames

Combinamos los tres DataFrames usando `merge()` sobre la columna `id_evento`, que es la clave común entre las tres tablas.

In [3]:
# Inner join: solo se conservan filas con id_evento presente en los tres DataFrames.
# Eventos sin aficionados o sin promociones quedarán fuera — decisión intencional.
df_merged = pd.merge(df_eventos, df_aficionados, on="id_evento")
df_merged = pd.merge(df_merged, df_promociones, on="id_evento")

print(f"DataFrame combinado: {df_merged.shape}")
print(df_merged.head())

DataFrame combinado: (39, 15)
   id_evento nombre_evento fecha_evento ubicación_evento  asistentes_totales  \
0         38     Evento 38   2023-02-07         Ciudad D                6614   
1          5      Evento 5   2023-01-05         Ciudad C                7647   
2          3      Evento 3   2023-01-03         Ciudad A                 254   
3          5      Evento 5   2023-01-05         Ciudad C                7647   
4         21     Evento 21   2023-01-21         Ciudad B                9634   

   id_aficionado nombre_aficionado  edad género ubicación_residencia  \
0             36     Aficionado 36    42      F             Ciudad D   
1             16     Aficionado 16    51      F             Ciudad D   
2              2      Aficionado 2    20      F             Ciudad D   
3             19     Aficionado 19    42      M             Ciudad D   
4             41     Aficionado 41    21      F             Ciudad A   

   id_promocion medio_publicitario  presupuesto fecha_in

## 4. Análisis de datos

Respondemos a las cuatro preguntas de negocio.

### 4.1 ¿Cuál es el evento con mayor asistencia total?

In [4]:
def get_evento_max_asistencia(df):
    max_asistentes = df["asistentes_totales"].max()
    return df[df["asistentes_totales"] == max_asistentes]

evento_max = get_evento_max_asistencia(df_eventos)
print("Evento con mayor asistencia:")
print(evento_max[["nombre_evento", "asistentes_totales"]])

Evento con mayor asistencia:
   nombre_evento  asistentes_totales
28     Evento 29                9879


### 4.2 ¿Qué rango de edad asiste más frecuentemente?

Usamos `pd.cut()` para agrupar a los aficionados en rangos de edad y contamos cuántos hay en cada grupo.

In [5]:
def get_rango_edad_frecuente(df):
    rangos    = [0, 18, 30, 45, 60, 100]
    etiquetas = ["<18", "18-30", "31-45", "46-60", ">60"]

    df = df.copy()
    df["rango_edad"] = pd.cut(df["edad"], bins=rangos, labels=etiquetas)
    return df["rango_edad"].value_counts().sort_values(ascending=False)

conteo_rangos = get_rango_edad_frecuente(df_aficionados)
print("Aficionados por rango de edad:")
print(conteo_rangos)
print(f"\nRango más frecuente: {conteo_rangos.index[0]}")

Aficionados por rango de edad:
rango_edad
18-30    20
31-45    15
46-60    13
<18       2
>60       0
Name: count, dtype: int64

Rango más frecuente: 18-30


### 4.3 ¿Qué medio publicitario ha generado mayor impacto?

Creamos la columna `impacto_promocion` dividiendo `asistentes_totales` entre `presupuesto`, y agrupamos por `medio_publicitario`.

In [6]:
def get_medio_mayor_impacto(df):
    df = df.copy()
    df["impacto_promocion"] = df["asistentes_totales"] / df["presupuesto"]
    impacto = df.groupby("medio_publicitario")["impacto_promocion"].mean().sort_values(ascending=False)
    return df, impacto

df_merged, impacto_por_medio = get_medio_mayor_impacto(df_merged)
print("Impacto medio por medio publicitario:")
print(impacto_por_medio)
print(f"\nMedio con mayor impacto: {impacto_por_medio.index[0]}")

Impacto medio por medio publicitario:
medio_publicitario
TV                0.504766
Radio             0.491687
Internet          0.451231
Redes Sociales    0.415709
Name: impacto_promocion, dtype: float64

Medio con mayor impacto: TV


### 4.4 ¿Cuál es la ubicación de residencia que más aficionados aporta?

Agrupamos por `ubicación_residencia` y contamos el número de aficionados de cada ciudad.

In [7]:
def get_ubicacion_mas_aficionados(df):
    return df.groupby("ubicación_residencia")["id_aficionado"].count().sort_values(ascending=False)

aficionados_por_ubicacion = get_ubicacion_mas_aficionados(df_aficionados)
print("Aficionados por ubicación de residencia:")
print(aficionados_por_ubicacion)
print(f"\nUbicación con más aficionados: {aficionados_por_ubicacion.index[0]}")

Aficionados por ubicación de residencia:
ubicación_residencia
Ciudad A    15
Ciudad B    12
Ciudad C    12
Ciudad D    11
Name: id_aficionado, dtype: int64

Ubicación con más aficionados: Ciudad A


## 5. Exportación de resultados

Guardamos el DataFrame combinado con todas las transformaciones en `reporte_eventos.csv`.

In [8]:
df_merged.to_csv(os.path.abspath("reporte_eventos.csv"), index=False)
print("Archivo exportado: reporte_eventos.csv")

# Resumen del DataFrame exportado
df_merged.info()

Archivo exportado: reporte_eventos.csv
<class 'pandas.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id_evento             39 non-null     int64  
 1   nombre_evento         39 non-null     str    
 2   fecha_evento          39 non-null     str    
 3   ubicación_evento      39 non-null     str    
 4   asistentes_totales    39 non-null     int64  
 5   id_aficionado         39 non-null     int64  
 6   nombre_aficionado     39 non-null     str    
 7   edad                  39 non-null     int64  
 8   género                39 non-null     str    
 9   ubicación_residencia  39 non-null     str    
 10  id_promocion          39 non-null     int64  
 11  medio_publicitario    39 non-null     str    
 12  presupuesto           39 non-null     int64  
 13  fecha_inicio          39 non-null     str    
 14  fecha_fin             39 non-null     str    
 1